# 🎵 Gospel Lyric Video Generator
## *Jesus Comin' By and By*

**Instructions:**
1. Run each cell top to bottom (click ▶ or press Shift+Enter)
2. Cell 1 installs libraries (~1 min)
3. Cell 2 builds the video (~3-5 min)
4. Cell 3 gives you a download link

> Works entirely in the browser — no desktop needed!

In [ ]:
# ── Cell 1: Install libraries ─────────────────────────────────────────────
!pip install -q moviepy Pillow numpy
print('✅ Libraries ready!')

In [ ]:
# ── Cell 2: Build the 3-minute gospel lyric video ─────────────────────────
from moviepy import (
    ColorClip, TextClip, CompositeVideoClip, VideoClip
)
from moviepy.video.fx import CrossFadeIn, CrossFadeOut
import numpy as np
from PIL import Image, ImageDraw, ImageFilter

# ── Lyrics timing (start, end, text, section) ──────────────────────────────
SECTIONS = [
    (0,   6,   "",                                                          "intro"),
    # Verse 1
    (6,   12,  "Last I saw that star shine yonder in the sky,",             "verse"),
    (12,  18,  "Burnin' bright like a promise that will never die,",        "verse"),
    (18,  24,  "Pointed straight to a Savior born for you and I,",          "verse"),
    (24,  32,  "Oh I felt His Spirit whisper,\n\"He's comin' by and by.\"", "verse"),
    # Verse 2
    (32,  38,  "Through the trials and the valleys, through the tears I've cried,", "verse"),
    (38,  44,  "There's a hope that keeps me steady deep inside,",          "verse"),
    (44,  50,  "Ain't no grave gonna hold me, ain't no shadow gonna hide,", "verse"),
    (50,  58,  "When I hear that trumpet sound,\nHe's comin' by and by.",  "verse"),
    # Chorus 1
    (58,  63,  "Hallelujah!\nHe's comin' for the ready,",                  "chorus"),
    (63,  67,  "Lift your voice and testify!",                              "chorus"),
    (67,  72,  "Saints are shoutin', hearts are steady,",                   "chorus"),
    (72,  78,  "Jesus Christ is comin' by and by!",                        "chorus"),
    (78,  83,  "Oh the heavens will open wide,",                            "chorus"),
    (83,  88,  "And we'll meet Him in the sky,",                            "chorus"),
    (88,  96,  "Hallelujah! Hallelujah!\nJesus comin' by and by!",         "chorus"),
    # Verse 3
    (96,  102, "I can see that eastern sky begin to glow,",                 "verse"),
    (102, 108, "Feel the fire of revival in my soul,",                      "verse"),
    (108, 114, "Every burden gonna vanish, every knee will bow,",           "verse"),
    (114, 122, "And the King of all creation's\ncomin' for me now.",       "verse"),
    # Bridge
    (122, 128, "Oh can't you hear the angels singin'?",                     "bridge"),
    (128, 134, "Oh can't you feel redemption ringin'?",                     "bridge"),
    (134, 142, "Every chain is breakin',\nevery heart awakenin',",          "bridge"),
    (142, 150, "Glory to the Lamb!",                                        "bridge"),
    # Chorus 2
    (150, 155, "Hallelujah!\nHe's comin' for the ready,",                  "chorus"),
    (155, 159, "Lift your voice and testify!",                              "chorus"),
    (159, 164, "Saints are shoutin', hearts are steady,",                   "chorus"),
    (164, 170, "Jesus Christ is comin' by and by!",                        "chorus"),
    (170, 175, "Oh the heavens will open wide,",                            "chorus"),
    (175, 179, "And we'll meet Him in the sky,",                            "chorus"),
    (179, 184, "Hallelujah! Hallelujah!\nJesus comin' by and by!",         "chorus"),
    # Outro
    (184, 188, "By and by…  (He's comin')",                                "outro"),
    (188, 192, "By and by…  (Oh yes He is)",                               "outro"),
    (192, 198, "Hallelujah, hallelujah,",                                   "outro"),
    (198, 207, "Jesus comin' by and by!",                                   "outro"),
    (207, 213, "",                                                          "intro"),
]

TARGET = 180   # clamp to 3 minutes
W, H   = 1920, 1080
FPS    = 24

SECTION_COLORS = {
    "verse":  (255, 240, 200),
    "chorus": (255, 220, 50),
    "bridge": (200, 230, 255),
    "outro":  (255, 200, 120),
    "intro":  (255, 255, 255),
}

BG_TOP    = (10, 18, 60)
BG_BOTTOM = (40, 10, 80)

# ── Background helpers ─────────────────────────────────────────────────────
def make_star_overlay():
    img  = Image.new("RGBA", (W, H), (0, 0, 0, 0))
    draw = ImageDraw.Draw(img)
    rng  = np.random.default_rng(42)
    n    = 350
    xs   = rng.integers(0, W, n)
    ys   = rng.integers(0, H // 2, n)
    rs   = rng.choice([1, 1, 2, 2, 3], size=n)
    als  = rng.integers(120, 255, n)
    for x, y, r, a in zip(xs, ys, rs, als):
        draw.ellipse([x-r, y-r, x+r, y+r], fill=(255, 255, 240, int(a)))
    return np.array(img)

def make_cross_overlay():
    img  = Image.new("RGBA", (W, H), (0, 0, 0, 0))
    draw = ImageDraw.Draw(img)
    cx, cy = int(W * 0.80), int(H * 0.42)
    t = 22
    draw.rectangle([cx-t//2, cy-110, cx+t//2, cy+110], fill=(255,230,120,60))
    draw.rectangle([cx-110,  cy-t//2, cx+110, cy+t//2], fill=(255,230,120,60))
    pil = Image.fromarray(np.array(img))
    pil = pil.filter(ImageFilter.GaussianBlur(radius=18))
    return np.array(pil)

_stars = make_star_overlay()
_cross = make_cross_overlay()

def blend(base, overlay):
    a   = overlay[:,:,3:4].astype(float) / 255.0
    rgb = overlay[:,:,:3].astype(float)
    return np.clip(base.astype(float)*(1-a) + rgb*a, 0, 255).astype(np.uint8)

def bg_frame(t):
    phase = t / TARGET
    top   = np.clip(np.array(BG_TOP,    float) + phase*[5,15,30],  0,255)
    bot   = np.clip(np.array(BG_BOTTOM, float) + phase*[20,0,10],  0,255)
    rows  = np.linspace(0, 1, H)[:, None]
    frame = (top*(1-rows) + bot*rows).astype(np.uint8)
    frame = np.broadcast_to(frame, (H, W, 3)).copy()
    frame = blend(frame, _stars)
    frame = blend(frame, _cross)
    return frame

# ── Lyric clip builder ─────────────────────────────────────────────────────
def lyric_clip(text, start, end, section):
    start, end = min(start, TARGET), min(end, TARGET)
    dur = end - start
    if dur <= 0 or not text.strip():
        return None
    color = "#{:02X}{:02X}{:02X}".format(*SECTION_COLORS.get(section,(255,255,255)))
    mlen  = max(len(l) for l in text.split("\n"))
    fsize = 72 if mlen<=30 else 60 if mlen<=45 else 52
    clip  = TextClip(
        text=text, font_size=fsize, color=color,
        font="DejaVu-Sans-Bold", text_align="center",
        method="caption", size=(int(W*0.85), None),
        stroke_color="#000000", stroke_width=3,
    )
    fade = min(0.6, dur*0.2)
    clip = clip.with_effects([CrossFadeIn(fade), CrossFadeOut(fade)])
    clip = clip.with_duration(dur).with_start(start)
    y = int(H*0.38) if section=="chorus" else int(H*0.42) if section=="bridge" else int(H*0.50) if section=="outro" else int(H*0.58)
    return clip.with_position(("center", y))

# ── Label clip builder ─────────────────────────────────────────────────────
def label_clip(text, start, end):
    start, end = min(start, TARGET), min(end, TARGET)
    dur = end - start
    if dur <= 0: return None
    clip = TextClip(text=text, font_size=28, color="#AAAACC",
                    font="DejaVu-Sans", method="label")
    clip = clip.with_effects([CrossFadeIn(0.4), CrossFadeOut(0.4)])
    return clip.with_duration(dur).with_start(start).with_position((60,40))

# ── Assemble ───────────────────────────────────────────────────────────────
print("Building background…")
bg = VideoClip(bg_frame, duration=TARGET).with_fps(FPS)

print("Building title card…")
t_main = (TextClip(text="Jesus Comin' By and By", font_size=90,
                   color="#FFD700", font="DejaVu-Sans-Bold", method="label",
                   stroke_color="#000000", stroke_width=4)
          .with_duration(6).with_start(0)
          .with_position(("center", int(H*0.38)))
          .with_effects([CrossFadeIn(1.0), CrossFadeOut(1.0)]))

t_sub  = (TextClip(text="A Gospel Celebration", font_size=44,
                   color="#FFFFFF", font="DejaVu-Sans", method="label",
                   stroke_color="#000000", stroke_width=2)
          .with_duration(6).with_start(0)
          .with_position(("center", int(H*0.54)))
          .with_effects([CrossFadeIn(1.5), CrossFadeOut(1.0)]))

print("Building lyric clips…")
lyrics = [c for s in SECTIONS if (c := lyric_clip(*s))]

labels = [
    label_clip("♪ Verse 1",  6,   32),
    label_clip("♪ Verse 2",  32,  58),
    label_clip("✦ Chorus",   58,  96),
    label_clip("♪ Verse 3",  96,  122),
    label_clip("♩ Bridge",   122, 150),
    label_clip("✦ Chorus",   150, 184),
    label_clip("♪ Outro",    184, TARGET),
]
labels = [l for l in labels if l]

print("Compositing…")
final = CompositeVideoClip(
    [bg, t_main, t_sub] + labels + lyrics,
    size=(W, H)
).with_duration(TARGET).with_fps(FPS)

print("Rendering video — this takes a few minutes…")
final.write_videofile(
    "gospel_video.mp4",
    fps=FPS,
    codec="libx264",
    audio=False,
    preset="fast",
    ffmpeg_params=["-crf", "23"],
    logger="bar",
)
print("✅ Done! gospel_video.mp4 is ready to download.")

In [ ]:
# ── Cell 3: Download the finished video ───────────────────────────────────
from google.colab import files
files.download('gospel_video.mp4')
print('Download started — check your iPhone Downloads folder!')